# 01 — Ingest, validate, clean, enrich (Part 1)
CareLoop Clinics · No-Show Guard. Tasks 1.1–1.6.

The three original files live in `data/` and are never edited. In Colab, upload the `data/` folder (or the three CSVs into a folder named `data`) before running.

In [1]:
import csv, json
import numpy as np
import pandas as pd
import requests

DATA = "data/"

## Task 1.1 — Read the file with the `csv` module only
`parse_row` turns the numeric text into numbers. A blank cell cannot become an `int`/`float`, so it raises `ValueError` and the row is rejected. Note that this is only a **type** check: an age of 250 converts fine and is accepted.

In [2]:
def parse_row(row):
    """Return a copy of the row with numeric fields converted. Raises ValueError on bad input."""
    out = dict(row)
    out["age"] = int(row["age"])
    out["fee_inr"] = int(row["fee_inr"])
    out["prior_no_shows"] = int(row["prior_no_shows"])
    out["distance_km"] = float(row["distance_km"])
    return out

accepted, rejected = [], []
with open(DATA + "appointments.csv", newline="") as f:
    reader = csv.DictReader(f)
    print("Header:", reader.fieldnames)
    rows = list(reader)
print("Rows read:", len(rows))

for row in rows:
    try:
        accepted.append(parse_row(row))
    except ValueError as e:
        rejected.append({"row": row, "error": str(e)})

print("Accepted:", len(accepted))
print("Rejected:", len(rejected))
print("  blank age:        ", sum(r["row"]["age"].strip() == "" for r in rejected))
print("  blank distance_km:", sum(r["row"]["distance_km"].strip() == "" for r in rejected))
print("Accepted rows with impossible ages (<1 or >100):", sum(not 1 <= r["age"] <= 100 for r in accepted))

with open("rejected_rows.json", "w") as f:
    json.dump(rejected, f, indent=2)
print("Saved rejected_rows.json")

Header: ['appointment_id', 'patient_id', 'clinic_id', 'booked_on', 'appt_date', 'age', 'gender', 'distance_km', 'fee_inr', 'booking_channel', 'visit_type', 'reminder_sent', 'prior_no_shows', 'status']
Rows read: 1020
Accepted: 988
Rejected: 32
  blank age:         12
  blank distance_km: 20
Accepted rows with impossible ages (<1 or >100): 5
Saved rejected_rows.json


## Task 1.2 — Inspect and clean, pass 1
This starts again from the original file using pandas. Task 1.1's accepted/rejected lists are not used here.

In [3]:
df = pd.read_csv(DATA + "appointments.csv")
print("Shape:", df.shape)
df.info()
print("\nNulls per column:")
print(df.isnull().sum())
print("\nExact duplicate rows:", df.duplicated().sum())

Shape: (1020, 14)
<class 'pandas.DataFrame'>
RangeIndex: 1020 entries, 0 to 1019
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   appointment_id   1020 non-null   str    
 1   patient_id       1020 non-null   str    
 2   clinic_id        1020 non-null   str    
 3   booked_on        1020 non-null   str    
 4   appt_date        1019 non-null   str    
 5   age              1008 non-null   float64
 6   gender           1020 non-null   str    
 7   distance_km      1000 non-null   float64
 8   fee_inr          1020 non-null   int64  
 9   booking_channel  1020 non-null   str    
 10  visit_type       1020 non-null   str    
 11  reminder_sent    1020 non-null   str    
 12  prior_no_shows   1020 non-null   int64  
 13  status           1016 non-null   str    
dtypes: float64(2), int64(2), str(10)
memory usage: 111.7 KB

Nulls per column:
appointment_id      0
patient_id          0
clinic_id           0
book

In [4]:
print("Rows at start:", len(df))

# (a) drop exact duplicates
df = df.drop_duplicates()
print("(a) after dropping duplicates:", len(df))

# (b) tidy booking_channel
print("    booking_channel raw spellings:", df["booking_channel"].nunique(), sorted(df["booking_channel"].unique()))
df["booking_channel"] = df["booking_channel"].str.strip().str.title()
print("(b) booking_channel after cleaning:", df["booking_channel"].nunique(), sorted(df["booking_channel"].unique()), "| rows:", len(df))

# (c) convert dates; unreadable appt_date values become NaT
bad_before = df["appt_date"].isna().sum()
df["appt_date"] = pd.to_datetime(df["appt_date"], errors="coerce")
df["booked_on"] = pd.to_datetime(df["booked_on"])
print("(c) appt_date missing after conversion:", df["appt_date"].isna().sum(),
      f"({df['appt_date'].isna().sum() - bad_before} newly unreadable) | rows:", len(df))

# (d) drop rows with no usable appt_date or status
print("    missing status:", df["status"].isna().sum())
df = df.dropna(subset=["appt_date", "status"])
print("(d) after dropping missing appt_date / status:", len(df))

Rows at start: 1020
(a) after dropping duplicates: 1000
    booking_channel raw spellings: 16 [' App', ' phone', ' walk-in', 'APP ', 'App', 'App ', 'PHONE', 'Phone', 'Phone ', 'WALK-IN', 'Walk-In', 'Walk-in', 'Walk-in ', 'app', 'phone', 'walk-in']
(b) booking_channel after cleaning: 3 ['App', 'Phone', 'Walk-In'] | rows: 1000
(c) appt_date missing after conversion: 6 (5 newly unreadable) | rows: 1000
    missing status: 4
(d) after dropping missing appt_date / status: 990


## Task 1.3 — Clean, pass 2: validity, gaps and outliers
Order: invalid → missing, fill gaps, *then* compute the quartiles and fences on the filled columns.

In [5]:
# (a) validity: impossible ages become missing
invalid_age = (df["age"] < 1) | (df["age"] > 100)
print("(a) invalid ages:", invalid_age.sum(), "->", sorted(df.loc[invalid_age, "age"].tolist()))
df.loc[invalid_age, "age"] = np.nan
print("    missing ages now:", df["age"].isna().sum())

# (b) fill gaps with medians
age_med = df["age"].median()
dist_med = df["distance_km"].median()
print("(b) median age:", age_med, "| median distance_km:", dist_med)
df["age"] = df["age"].fillna(age_med).round().astype(int)
df["distance_km"] = df["distance_km"].fillna(dist_med)

# (c) IQR capping
for col in ["distance_km", "fee_inr"]:
    q1, q3 = df[col].quantile(0.25), df[col].quantile(0.75)
    iqr = q3 - q1
    low, high = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    n_high = (df[col] > high).sum()
    n_low = (df[col] < low).sum()
    df[col] = np.clip(df[col], low, high)
    print(f"(c) {col}: Q1={q1}, Q3={q3}, IQR={iqr}, fences=[{low}, {high}] -> capped {n_high + n_low} "
          f"({n_high} high, {n_low} low)")

# (d) rounding
df["distance_km"] = df["distance_km"].round(1)
df["fee_inr"] = df["fee_inr"].round().astype(int)
print("(d) largest fee after capping:", df["fee_inr"].max(), "| largest distance:", df["distance_km"].max())

(a) invalid ages: 5 -> [-4.0, 0.0, 150.0, 199.0, 250.0]
    missing ages now: 17
(b) median age: 48.0 | median distance_km: 8.25
(c) distance_km: Q1=4.625, Q3=13.0, IQR=8.375, fences=[-7.9375, 25.5625] -> capped 26 (26 high, 0 low)
(c) fee_inr: Q1=500.0, Q3=787.5, IQR=287.5, fences=[68.75, 1218.75] -> capped 3 (3 high, 0 low)
(d) largest fee after capping: 1219 | largest distance: 25.6


## Task 1.4 — Features and a NumPy summary

In [6]:
df["lead_days"] = (df["appt_date"] - df["booked_on"]).dt.days
df["no_show"] = (df["status"] == "No-Show").astype(int)
print("lead_days range:", df["lead_days"].min(), "to", df["lead_days"].max())
print(f"No-shows: {df['no_show'].sum()} of {len(df)} ({round(100 * df['no_show'].mean(), 1)}%)")

X = df[["age", "distance_km", "fee_inr", "lead_days"]].to_numpy()

# (a) column means
print("(a) column means [age, distance_km, fee_inr, lead_days]:", np.round(X.mean(axis=0), 2))

# (b) long lead times
mask = X[:, 3] > 14
print(f"(b) rows with lead_days > 14: {mask.sum()} ({round(100 * mask.mean(), 1)}%)")

# (c) z-scores of distance_km (np.std = population SD)
x = X[:, 1]
z = (x - np.mean(x)) / np.std(x)
print("(c) rows with |z| > 2 for distance_km:", (np.abs(z) > 2).sum())

lead_days range: 1 to 45
No-shows: 218 of 990 (22.0%)
(a) column means [age, distance_km, fee_inr, lead_days]: [ 48.17   9.5  635.71  11.82]
(b) rows with lead_days > 14: 296 (29.9%)
(c) rows with |z| > 2 for distance_km: 53


## Task 1.5 — Enrich the clinics from Open-Meteo
Each clinic gets up to two tries, with `timeout=10`. If both tries fail, the status is not 200, or `hourly` is missing, `temp_c` is stored as `None` and the reason is printed. No temperature is ever typed in by hand.

In [7]:
clinics = pd.read_csv(DATA + "clinics.csv")
URL = "https://api.open-meteo.com/v1/forecast"

def fetch_temp(lat, lon):
    """Return (temp_c, None) on success or (None, reason) on failure."""
    params = {"latitude": lat, "longitude": lon, "hourly": "temperature_2m", "forecast_days": 1}
    resp = requests.get(URL, params=params, timeout=10)
    if resp.status_code != 200:
        return None, f"status {resp.status_code}"
    data = resp.json()
    if "hourly" not in data or not data["hourly"].get("temperature_2m"):
        return None, "no hourly temperature in reply"
    return data["hourly"]["temperature_2m"][0], None

temps = []
for _, c in clinics.iterrows():
    temp, reason = None, None
    for attempt in (1, 2):                     # first try + one retry
        try:
            temp, reason = fetch_temp(c["latitude"], c["longitude"])
        except requests.exceptions.RequestException as e:
            temp, reason = None, f"{type(e).__name__}"
        if temp is not None:
            break
    if temp is None:
        print(f"FAILED: {c['clinic_name']} — {reason} (after 2 attempts); temp_c left blank")
    else:
        print(f"OK: {c['clinic_name']}: {temp} °C")
    temps.append(temp)

clinics["temp_c"] = temps
clinics.to_csv("clinics_enriched.csv", index=False)
print("\nSaved clinics_enriched.csv:", clinics.shape)
print(clinics)

FAILED: Kothrud Family Clinic — ProxyError (after 2 attempts); temp_c left blank
FAILED: Hadapsar Care Centre — ProxyError (after 2 attempts); temp_c left blank
FAILED: Sitabuldi Health Point — ProxyError (after 2 attempts); temp_c left blank
FAILED: Malviya Nagar Clinic — ProxyError (after 2 attempts); temp_c left blank
FAILED: Vaishali Nagar Clinic — ProxyError (after 2 attempts); temp_c left blank
FAILED: Edappally Medical Centre — ProxyError (after 2 attempts); temp_c left blank
FAILED: Vijay Nagar Clinic — ProxyError (after 2 attempts); temp_c left blank
FAILED: Gomti Nagar Care Point — ProxyError (after 2 attempts); temp_c left blank

Saved clinics_enriched.csv: (8, 7)
  clinic_id               clinic_name     city  ... latitude  longitude  temp_c
0       C01     Kothrud Family Clinic     Pune  ...  18.5074    73.8077    None
1       C02      Hadapsar Care Centre     Pune  ...  18.5089    73.9260    None
2       C03    Sitabuldi Health Point   Nagpur  ...  21.1458    79.0882    N

## Task 1.6 — Export the clean table and the SQL load script

In [8]:
COLS = ["appointment_id", "patient_id", "clinic_id", "booked_on", "appt_date", "age", "gender",
        "distance_km", "fee_inr", "booking_channel", "visit_type", "reminder_sent", "prior_no_shows",
        "status", "lead_days", "no_show"]

clean = df.sort_values("appointment_id")[COLS].copy()
clean["booked_on"] = clean["booked_on"].dt.strftime("%Y-%m-%d")
clean["appt_date"] = clean["appt_date"].dt.strftime("%Y-%m-%d")

print("appointment_id unique:", clean["appointment_id"].is_unique, "| distinct:", clean["appointment_id"].nunique())
print("Missing values across the 16 columns:", int(clean.isnull().sum().sum()))
assert clean["appointment_id"].is_unique and clean.isnull().sum().sum() == 0

clean.to_csv("appointments_clean.csv", index=False)
print("Saved appointments_clean.csv:", clean.shape)

appointment_id unique: True | distinct: 990
Missing values across the 16 columns: 0
Saved appointments_clean.csv: (990, 16)


In [9]:
TEXT_COLS = {"appointment_id", "patient_id", "clinic_id", "booked_on", "appt_date", "gender",
             "booking_channel", "visit_type", "reminder_sent", "status",
             "clinic_name", "city", "tier"}

def sql_value(col, v):
    if pd.isna(v):
        return "NULL"
    return f"'{v}'" if col in TEXT_COLS else f"{v}"

clinics_out = pd.read_csv("clinics_enriched.csv")
n_clin = n_appt = 0
with open("load_data.sql", "w") as f:
    f.write("-- Generated by 01_ingest_clean.ipynb (Task 1.6). Clinics first: appointments reference them.\n")
    for _, r in clinics_out.iterrows():
        vals = ",".join(sql_value(c, r[c]) for c in clinics_out.columns)
        f.write(f"INSERT INTO clinics VALUES ({vals});\n")
        n_clin += 1
    for _, r in clean.iterrows():
        vals = ",".join(sql_value(c, r[c]) for c in COLS)
        f.write(f"INSERT INTO appointments VALUES ({vals});\n")
        n_appt += 1

print(f"load_data.sql: {n_clin} clinic + {n_appt} appointment = {n_clin + n_appt} INSERT statements")
with open("load_data.sql") as f:
    lines = f.read().splitlines()
print(lines[1]); print(lines[9])

load_data.sql: 8 clinic + 990 appointment = 998 INSERT statements
INSERT INTO clinics VALUES ('C01','Kothrud Family Clinic','Pune','Metro',18.5074,73.8077,NULL);
INSERT INTO appointments VALUES ('A0001','P509','C05','2025-01-24','2025-01-25',60,'F',18.0,850,'Phone','Consultation','Yes',1,'No-Show',1,1);
